In [1]:
import pandas as pd

nav_history = pd.read_csv("02_nav_history.csv")
investor_transactions = pd.read_csv("08_investor_transactions.csv")
scheme_performance = pd.read_csv("07_scheme_performance.csv")

print("NAV History Columns:")
print(nav_history.columns.tolist())

print("\nInvestor Transactions Columns:")
print(investor_transactions.columns.tolist())

print("\nScheme Performance Columns:")
print(scheme_performance.columns.tolist())

NAV History Columns:
['amfi_code', 'date', 'nav']

Investor Transactions Columns:
['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']

Scheme Performance Columns:
['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']


In [2]:
import pandas as pd
import os

# Create cleaned_data folder
os.makedirs("cleaned_data", exist_ok=True)

# Load file
nav = pd.read_csv("02_nav_history.csv")

# Convert date to datetime
nav['date'] = pd.to_datetime(nav['date'], errors='coerce')

# Convert nav to numeric
nav['nav'] = pd.to_numeric(nav['nav'], errors='coerce')

# Sort by fund code and date
nav = nav.sort_values(['amfi_code', 'date'])

# Remove duplicates
nav = nav.drop_duplicates(subset=['amfi_code', 'date'])

# Forward-fill NAV within each fund
nav['nav'] = nav.groupby('amfi_code')['nav'].ffill()

# Remove invalid NAV values
nav = nav[nav['nav'] > 0]

print("NAV History Shape:", nav.shape)

# Save
nav.to_csv("cleaned_data/nav_history_clean.csv", index=False)

NAV History Shape: (46000, 3)


In [4]:
perf = pd.read_csv("07_scheme_performance.csv")

# Convert return columns to numeric
return_cols = [
    'return_1yr_pct',
    'return_3yr_pct',
    'return_5yr_pct',
    'benchmark_3yr_pct'
]

for col in return_cols:
    perf[col] = pd.to_numeric(
        perf[col],
        errors='coerce'
    )

# Convert expense ratio
perf['expense_ratio_pct'] = pd.to_numeric(
    perf['expense_ratio_pct'],
    errors='coerce'
)

# Flag unusual expense ratios
expense_flags = perf[
    (perf['expense_ratio_pct'] < 0.1) |
    (perf['expense_ratio_pct'] > 2.5)
]

print("Expense Ratio Issues:")
print(expense_flags[
    ['scheme_name', 'expense_ratio_pct']
])

print("Rows with unusual expense ratios:",
      len(expense_flags))

# Save
perf.to_csv(
    "cleaned_data/scheme_performance_clean.csv",
    index=False
)

Expense Ratio Issues:
Empty DataFrame
Columns: [scheme_name, expense_ratio_pct]
Index: []
Rows with unusual expense ratios: 0


In [5]:
txn = pd.read_csv("08_investor_transactions.csv")

# Fix date format
txn['transaction_date'] = pd.to_datetime(
    txn['transaction_date'],
    errors='coerce'
)

# Standardize transaction type
txn['transaction_type'] = (
    txn['transaction_type']
    .astype(str)
    .str.strip()
    .str.title()
)

txn['transaction_type'] = txn['transaction_type'].replace({
    'Sip': 'SIP',
    'Lumpsum': 'Lumpsum',
    'Redemption': 'Redemption'
})

# Keep only valid transaction types
txn = txn[
    txn['transaction_type'].isin(
        ['SIP', 'Lumpsum', 'Redemption']
    )
]

# Amount must be positive
txn = txn[txn['amount_inr'] > 0]

# Check KYC values
print("KYC Status Values:")
print(txn['kyc_status'].unique())

print("Transaction Shape:", txn.shape)

# Save
txn.to_csv(
    "cleaned_data/investor_transactions_clean.csv",
    index=False
)

KYC Status Values:
['Verified' 'Pending']
Transaction Shape: (32778, 13)


In [6]:
print("Missing values in NAV:")
print(nav.isnull().sum())

print("\nMissing values in Transactions:")
print(txn.isnull().sum())

print("\nMissing values in Performance:")
print(perf.isnull().sum())

Missing values in NAV:
amfi_code    0
date         0
nav          0
dtype: int64

Missing values in Transactions:
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64

Missing values in Performance:
amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64


In [7]:
print("Duplicate NAV records:",
      nav.duplicated(subset=['amfi_code','date']).sum())

print("Invalid NAV values:",
      (nav['nav'] <= 0).sum())

Duplicate NAV records: 0
Invalid NAV values: 0


In [8]:
print(txn['transaction_type'].unique())

['SIP' 'Redemption' 'Lumpsum']


In [9]:
print((txn['amount_inr'] <= 0).sum())

0


In [10]:
print(txn['kyc_status'].unique())

['Verified' 'Pending']


In [11]:
print(perf.dtypes)
expense_flags = perf[
    (perf['expense_ratio_pct'] < 0.1) |
    (perf['expense_ratio_pct'] > 2.5)
]

print("Rows with unusual expense ratios:",
      len(expense_flags))

amfi_code               int64
scheme_name            object
fund_house             object
category               object
plan                   object
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade             object
dtype: object
Rows with unusual expense ratios: 0
